# 03 – Feature Engineering

Goal: Build the feature matrix for XGBoost training.

Features constructed:
1. **Macro lag features** – lags 1–6 for all four macro indicators (non-stationary series as first differences)
2. **Calendar features** – month, quarter, year, is_q4
3. **Autoregressive features** – revenue at lag 1, 2, 3 months
4. **Rolling window features** – 3M and 6M rolling mean and std of revenue

Output: `data/processed/feature_matrix.csv` — one row per product × month, with a `product` column for flexible filtering.

**Note:** XGBoost training targets Compressors/US (B3). Accessories is included in the matrix for completeness and can be filtered out at training time. Lag selection is intentionally left to XGBoost feature importance and SHAP — CCF results from 02 serve as a plausibility check only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'

# Non-stationary series (from ADF results in 02) — will be first-differenced
NON_STATIONARY = ['de_production_index', 'us_durable_goods_orders_musd', 'us_production_index']
STATIONARY     = ['de_orders_index']
MACRO_COLS     = STATIONARY + NON_STATIONARY

LAG_RANGE = range(1, 7)  # lags 1–6

---
## 1. Load Revenue Data

In [ ]:
# Load Net Value per product
net_value = pd.read_excel(DATA_RAW + 'DGE_Revenue_Planning_Actuals_Net_Value_US.xlsx', sheet_name=0)
net_value = net_value.rename(columns={'Unnamed: 4': 'net_value_usd'})
net_value['date'] = pd.to_datetime(net_value['Date'].astype(str), format='%Y%m')
net_value = net_value[['date', 'Product', 'net_value_usd']].dropna().sort_values('date').reset_index(drop=True)

print('Revenue shape:', net_value.shape)
print('Products:', net_value['Product'].unique())
print('Date range:', net_value['date'].min(), '–', net_value['date'].max())
net_value.head()

---
## 2. Load & Prepare Macro Features

In [ ]:
# Load processed macro data from 02_macro_data.ipynb
macro = pd.read_csv(DATA_PROCESSED + 'macro_features.csv', parse_dates=['date'])

print('Macro shape:', macro.shape)
print('Date range:', macro['date'].min(), '–', macro['date'].max())
macro.head()

In [ ]:
# Apply first differences to non-stationary series
# The differenced columns are named with a '_diff' suffix to distinguish from levels
macro_prep = macro.copy()

for col in NON_STATIONARY:
    macro_prep[f'{col}_diff'] = macro_prep[col].diff()

# Drop original non-stationary level columns — use diff versions for lag features
macro_prep = macro_prep.drop(columns=NON_STATIONARY)

print('Prepared macro columns:', list(macro_prep.columns))
macro_prep.head()

---
## 3. Merge Revenue + Macro

In [ ]:
# Merge on date — left join keeps all revenue months
# Revenue starts 2022-10, macro starts 2018-01 → all revenue dates will find a macro match
df = net_value.merge(macro_prep, on='date', how='left').sort_values(['Product', 'date']).reset_index(drop=True)

print('Merged shape:', df.shape)
print('Missing values after merge:')
print(df.isnull().sum())
df.head()

---
## 4. Macro Lag Features (Lags 1–6)

Lag features are created per product group to avoid cross-product data leakage.
Each macro column gets 6 lagged versions: `{col}_lag_1` … `{col}_lag_6`.

In [ ]:
# Macro columns available after differencing
macro_feature_cols = [c for c in df.columns if c not in ['date', 'Product', 'net_value_usd']]
print('Macro feature columns:', macro_feature_cols)

# Create lag features per product group
lag_frames = []
for product, group in df.groupby('Product'):
    group = group.sort_values('date').copy()
    for col in macro_feature_cols:
        for lag in LAG_RANGE:
            group[f'{col}_lag_{lag}'] = group[col].shift(lag)
    lag_frames.append(group)

df = pd.concat(lag_frames).sort_values(['Product', 'date']).reset_index(drop=True)

# Drop the unlagged macro columns — only lags are used as features
df = df.drop(columns=macro_feature_cols)

print('\nShape after macro lag features:', df.shape)
lag_cols = [c for c in df.columns if '_lag_' in c]
print(f'Lag feature columns created: {len(lag_cols)}')

---
## 5. Calendar Features

In [ ]:
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['year']    = df['date'].dt.year
df['is_q4']   = (df['quarter'] == 4).astype(int)  # B2B year-end effect

print('Calendar features added: month, quarter, year, is_q4')
df[['date', 'Product', 'month', 'quarter', 'year', 'is_q4']].head(8)

---
## 6. Autoregressive Features (Revenue Lag 1–3)

Computed per product group to avoid leakage between products.

In [ ]:
ar_frames = []
for product, group in df.groupby('Product'):
    group = group.sort_values('date').copy()
    for lag in [1, 2, 3]:
        group[f'revenue_lag_{lag}'] = group['net_value_usd'].shift(lag)
    ar_frames.append(group)

df = pd.concat(ar_frames).sort_values(['Product', 'date']).reset_index(drop=True)

print('AR features added: revenue_lag_1, revenue_lag_2, revenue_lag_3')
df[['date', 'Product', 'net_value_usd', 'revenue_lag_1', 'revenue_lag_2', 'revenue_lag_3']].head(8)

---
## 7. Rolling Window Features (3M and 6M)

Computed per product group. `min_periods=1` avoids NaN for the first months.

In [ ]:
roll_frames = []
for product, group in df.groupby('Product'):
    group = group.sort_values('date').copy()
    # Shift by 1 before rolling to avoid using the current month's value (data leakage)
    rev_shifted = group['net_value_usd'].shift(1)
    group['revenue_rolling_3m_mean'] = rev_shifted.rolling(3, min_periods=1).mean()
    group['revenue_rolling_3m_std']  = rev_shifted.rolling(3, min_periods=1).std()
    group['revenue_rolling_6m_mean'] = rev_shifted.rolling(6, min_periods=1).mean()
    group['revenue_rolling_6m_std']  = rev_shifted.rolling(6, min_periods=1).std()
    roll_frames.append(group)

df = pd.concat(roll_frames).sort_values(['Product', 'date']).reset_index(drop=True)

print('Rolling features added: 3M and 6M mean + std (shifted by 1 to prevent leakage)')
df[['date', 'Product', 'net_value_usd',
    'revenue_rolling_3m_mean', 'revenue_rolling_3m_std',
    'revenue_rolling_6m_mean', 'revenue_rolling_6m_std']].head(8)

---
## 8. Final Feature Matrix

Drop rows with NaN values introduced by lagging (first few months per product).  
Print a summary of the final matrix.

In [ ]:
rows_before = len(df)
df_final = df.dropna().reset_index(drop=True)
rows_after = len(df_final)

print(f'Rows before dropna: {rows_before}')
print(f'Rows after  dropna: {rows_after}  ({rows_before - rows_after} dropped)')
print(f'\nFinal shape: {df_final.shape}')
print(f'Date range:  {df_final["date"].min().date()} – {df_final["date"].max().date()}')
print(f'Products:    {df_final["Product"].unique()}')
print(f'\nFeature count (excl. date, product, target): {len(df_final.columns) - 3}')
print('\nAll columns:')
for col in df_final.columns:
    print(f'  {col}')

In [ ]:
# Quick visual check: feature correlation heatmap (Compressors only)
comp = df_final[df_final['Product'] == 'Compressors'].drop(columns=['date', 'Product'])

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    comp.corr(),
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.6}
)
ax.set_title('Feature Correlation Matrix – Compressors', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Save Feature Matrix

In [ ]:
import os
os.makedirs(DATA_PROCESSED, exist_ok=True)

output_path = DATA_PROCESSED + 'feature_matrix.csv'
df_final.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {df_final.shape}')
df_final.head()